# EHR-M-GAN (Neo M3GAN) Training on Kaggle

This notebook is prepared for training the **neo_m3gan** model. It includes all necessary source files and handles directory setup for the Kaggle environment.

### Before you start:
1. **GPU Required**: Go to **Settings** (right sidebar) -> **Accelerator** -> Select **GPU T4 x2** (or P100).
2. **Internet Required**: Go to **Settings** -> **Internet** -> Select **On**.
3. **Add Data**: Click **+ Add Data** -> Search for your dataset **`mimic3`** and add it.

---

In [1]:
# 1. Create directory structure
import os
os.makedirs('Data/mimic', exist_ok=True)
os.makedirs('Output/checkpoint', exist_ok=True)
os.makedirs('logs', exist_ok=True)

# 2. Link Kaggle Data to local Data directory
dataset_root = '/kaggle/input/datasets/lmnhthng/mimic3'

for filename in ['vital_sign_24hrs.pkl', 'med_interv_24hrs.pkl']:
    src = os.path.join(dataset_root, filename)
    dst = os.path.join('Data/mimic', filename)
    if os.path.exists(src):
        if os.path.exists(dst): os.remove(dst)
        os.symlink(src, dst)
        print(f"Linked {filename}")
    else:
        print(f"WARNING: {filename} not found in {dataset_root}")

Linked vital_sign_24hrs.pkl
Linked med_interv_24hrs.pkl


In [2]:
%%writefile ultils.py
import torch, torch.nn.functional as F, numpy as np
def nt_xent_loss(out_1, out_2, temperature=1.0):
    batch_size = out_1.shape[0]
    out_1 = F.normalize(out_1, p=2, dim=-1)
    out_2 = F.normalize(out_2, p=2, dim=-1)
    out = torch.cat([out_1, out_2], dim=0)
    cov = torch.mm(out, out.t().contiguous())
    sim = torch.exp(cov / temperature)
    mask = ~torch.eye(2 * batch_size, device=out.device).bool()
    negatives = sim.masked_select(mask).view(2 * batch_size, -1)
    positives = torch.exp(torch.sum(out_1 * out_2, dim=-1) / temperature)
    positives = torch.cat([positives, positives], dim=0)
    return -torch.log(positives / (negatives.sum(dim=-1) + 1e-8) + 1e-8).mean()
def kl_divergence(mu, logvar): return -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1))
def feature_matching_loss(fk, rl):
    m_fk, m_rl = torch.mean(fk, 0), torch.mean(rl, 0)
    s_fk, s_rl = torch.sqrt(torch.var(fk, 0) + 1e-6), torch.sqrt(torch.var(rl, 0) + 1e-6)
    return torch.mean(torch.abs(m_fk - m_rl)) + torch.mean(torch.abs(s_fk - s_rl))
def renormlizer(d, ma, mi): return d * (ma + 1e-8) + mi
def np_rounding(p): return np.round(p)

Writing ultils.py


In [3]:
%%writefile networks.py
import torch, torch.nn as nn
from torch.nn.utils import spectral_norm
class TemporalSelfAttention(nn.Module):
    def __init__(self, hd):
        super().__init__()
        self.query, self.key, self.value = nn.Linear(hd, hd//8), nn.Linear(hd, hd//8), nn.Linear(hd, hd)
        self.gamma = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        b, t, d = x.size()
        pq, pk, pv = self.query(x).view(b,-1,t).permute(0,2,1), self.key(x).view(b,-1,t), self.value(x).view(b,-1,t)
        attn = torch.softmax(torch.bmm(pq, pk), -1)
        return self.gamma * torch.bmm(pv, attn.permute(0,2,1)).view(b,t,d) + x
class VAE_Encoder(nn.Module):
    def __init__(self, id, hd, ld, nl=3):
        super().__init__()
        self.lstm = nn.LSTM(id, hd, nl, batch_first=True, dropout=0.2 if nl>1 else 0)
        self.fc_mu, self.fc_logvar = nn.Linear(hd, ld), nn.Linear(hd, ld)
    def forward(self, x, h):
        o, nh = self.lstm(x, h)
        os = o.squeeze(1)
        m, l = torch.clamp(self.fc_mu(os), -20, 20), torch.clamp(self.fc_logvar(os), -20, 20)
        return m + torch.randn_like(m)*torch.exp(0.5*l), m, l, nh
class VAE_Decoder(nn.Module):
    def __init__(self, ld, hd, od, nl=3):
        super().__init__()
        self.lstm = nn.LSTM(ld, hd, nl, batch_first=True, dropout=0.2 if nl>1 else 0)
        self.fc_out = nn.Linear(hd, od)
    def forward(self, z, h):
        o, nh = self.lstm(z, h)
        lt = self.fc_out(o.squeeze(1))
        return torch.sigmoid(lt), lt, nh
class AutoregressiveVAE(nn.Module):
    def __init__(self, id, hd, ld, el, dl, ts):
        super().__init__()
        self.ts, self.id, self.hd = ts, id, hd
        self.encoder, self.decoder = VAE_Encoder(id*2, hd, ld, el), VAE_Decoder(ld, hd, id, dl)
    def forward(self, x):
        b, dev = x.size(0), x.device
        eh = (torch.zeros(self.encoder.lstm.num_layers, b, self.hd, device=dev), torch.zeros(self.encoder.lstm.num_layers, b, self.hd, device=dev))
        dh = (torch.zeros(self.decoder.lstm.num_layers, b, self.hd, device=dev), torch.zeros(self.decoder.lstm.num_layers, b, self.hd, device=dev))
        cp, res = torch.zeros(b, self.id, device=dev), [[] for _ in range(5)]
        for t in range(self.ts):
            xt = x[:, t, :]
            zt, mt, vt, eh = self.encoder(torch.cat([xt, xt-torch.sigmoid(cp)], 1).unsqueeze(1), eh)
            rt, lt, dh = self.decoder(zt.unsqueeze(1), dh); cp = lt
            for i, v in enumerate([rt.unsqueeze(1), lt.unsqueeze(1), mt.unsqueeze(1), vt.unsqueeze(1), zt.unsqueeze(1)]): res[i].append(v)
        return [torch.cat(r, 1) for r in res]
    def reconstruct_decoder(self, zs):
        b, dev = zs.size(0), zs.device
        dh = (torch.zeros(self.decoder.lstm.num_layers, b, self.hd, device=dev), torch.zeros(self.decoder.lstm.num_layers, b, self.hd, device=dev))
        rl, ll = [], []
        for t in range(self.ts):
            rt, lt, dh = self.decoder(zs[:, t, :].unsqueeze(1), dh)
            rl.append(rt.unsqueeze(1)); ll.append(lt.unsqueeze(1))
        return torch.cat(rl, 1), torch.cat(ll, 1)
class SequenceDiscriminator(nn.Module):
    def __init__(self, id, hd, ts, nl=3):
        super().__init__()
        self.lstm = nn.LSTM(id, hd, nl, batch_first=True, dropout=0.2 if nl>1 else 0)
        self.attn, self.fc = TemporalSelfAttention(hd), spectral_norm(nn.Linear(hd*ts, 1))
    def forward(self, x): 
        o, _ = self.lstm(x); ot = self.attn(o)
        return self.fc(ot.reshape(ot.size(0), -1)).squeeze(-1), ot
class BilateralLSTMCell(nn.Module):
    def __init__(self, id, hd):
        super().__init__()
        self.ln = nn.Linear(id + 2*hd, 4*hd, bias=False)
    def forward(self, x, hs, cs, hc):
        i, f, o, ct = self.ln(torch.cat([x, hs, hc], 1)).chunk(4, 1)
        cn = torch.sigmoid(f)*cs + torch.sigmoid(i)*torch.tanh(ct)
        return torch.sigmoid(o)*torch.tanh(cn), cn
class BilateralGenerator(nn.Module):
    def __init__(self, nd, hd, ld, nl=3):
        super().__init__()
        self.nl, self.hd = nl, hd
        self.cl = nn.ModuleList([BilateralLSTMCell(nd if i==0 else hd, hd) for i in range(nl)])
        self.fc_out = nn.Linear(hd, ld)
    def forward(self, ns, hc):
        b, t, dev = ns.size(0), ns.size(1), ns.device
        h, c = [torch.zeros(b, self.hd, device=dev) for _ in range(self.nl)], [torch.zeros(b, self.hd, device=dev) for _ in range(self.nl)]
        ol = []
        for t in range(t):
            xt = ns[:, t, :]
            for i in range(self.nl): h[i], c[i] = self.cl[i](xt, h[i], c[i], hc[i]); xt = h[i]
            ol.append(torch.sigmoid(self.fc_out(xt)).unsqueeze(1))
        return torch.cat(ol, 1), h
class JointGenerator(nn.Module):
    def __init__(self, cnd, dnd, hd, cl, dl, nl=3):
        super().__init__()
        self.nl, self.hd = nl, hd
        self.c_gen, self.d_gen = BilateralGenerator(cnd, hd, cl, nl), BilateralGenerator(dnd, hd, dl, nl)
        self.c_attn, self.d_attn = TemporalSelfAttention(hd), TemporalSelfAttention(hd)
    def forward(self, nc, nd):
        b, t, dev = nc.size(0), nc.size(1), nc.device
        ch, cc, dh, dc = [[torch.zeros(b, self.hd, device=dev) for _ in range(self.nl)] for _ in range(4)]
        cf, df = [], []
        for t in range(t):
            cx, dx = nc[:, t, :], nd[:, t, :]
            for i in range(self.nl): ch[i], cc[i] = self.c_gen.cl[i](cx, ch[i], cc[i], dh[i]); cx = ch[i]
            cf.append(cx.unsqueeze(1))
            for i in range(self.nl): dh[i], dc[i] = self.d_gen.cl[i](dx, dh[i], dc[i], ch[i]); dx = dh[i]
            df.append(dx.unsqueeze(1))
        return self.c_gen.fc_out(self.c_attn(torch.cat(cf, 1))), self.d_gen.fc_out(self.d_attn(torch.cat(df, 1)))

Writing networks.py


In [4]:
%%writefile metrics.py
import numpy as np
from sklearn.metrics.pairwise import rbf_kernel, euclidean_distances
import warnings
def discrete_probability_rmse(rd, fd): return np.sqrt(np.mean((np.mean(rd, (0,1)) - np.mean(fd, (0,1)))**2))
def mmd_rbf(rc, fc):
    x, y = rc.reshape(rc.shape[0], -1), fc.reshape(fc.shape[0], -1)
    dists = euclidean_distances(x, x)
    sig = np.median(dists[dists>0]) or 1.0
    gam = 1.0 / (2.0 * sig**2)
    return rbf_kernel(x, x, gam).mean() + rbf_kernel(y, y, gam).mean() - 2*rbf_kernel(x, y, gam).mean()
def pearson_correlation_error(r, f):
    rf, ff = r.reshape(-1, r.shape[2]), f.reshape(-1, f.shape[2])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return np.mean(np.abs(np.nan_to_num(np.corrcoef(rf, rowvar=False)) - np.nan_to_num(np.corrcoef(ff, rowvar=False))))
def evaluate_all(rc, fc, rd, fd):
    res = {'mmd':mmd_rbf(rc, fc), 'rmse':discrete_probability_rmse(rd, fd), 'corr_c':pearson_correlation_error(rc, fc), 'corr_d':pearson_correlation_error(rd, fd)}
    print(f"MMD: {res['mmd']:.5f} | RMSE: {res['rmse']:.5f} | CorrC: {res['corr_c']:.5f}")
    return res

Writing metrics.py


In [5]:
%%writefile visualise.py
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt, random, os
from ultils import renormlizer
def visualise_gan(rc, sc, rd, sd, inx, mav, miv, nd=12, np_=10, sp="logs/"):
    os.makedirs(sp, exist_ok=True); rc, sc = renormlizer(rc, mav, miv), renormlizer(sc, mav, miv)
    fig, ax = plt.subplots(4, nd, figsize=(4*nd, 16))
    ci, di = random.sample(range(rc.shape[2]), nd), random.sample(range(rd.shape[2]), nd)
    for i in range(nd):
        for j, d in enumerate([rc, sc]):
            a, df = ax[j, i], pd.DataFrame(d[random.sample(range(len(d)), np_), :, ci[i]])
            sns.lineplot(ax=a, data=df.T, palette='Greens' if j==0 else 'Reds', alpha=0.3, legend=False)
            sns.lineplot(ax=a, data=df.T.mean(0), color='black', linewidth=2)
            if j==1: a.set_ylim(ax[0, i].get_ylim())
        for j, d in enumerate([rd, sd]): sns.heatmap(d[random.sample(range(len(d)), np_), :, di[i]], ax=ax[j+2, i], cmap='Greens' if j==0 else 'Reds', cbar=False)
    plt.tight_layout(); fig.savefig(f"{sp}/vis_{inx}.png"); plt.close(fig)
def plot_metrics_trend(h, sp="logs/"):
    if not h['epochs']: return
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    for i, (k, c) in enumerate([('mmd', 'blue'), ('rmse', 'green'), ('corr_c', 'red')]): ax[i].plot(h['epochs'], h[k], marker='o', color=c); ax[i].set_title(k.upper())
    plt.tight_layout(); fig.savefig(f"{sp}/metrics.png"); plt.close(fig)

Writing visualise.py


In [10]:
%%writefile trainer.py
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F, numpy as np, os
from torch.utils.data import DataLoader
from tqdm import tqdm
from networks import AutoregressiveVAE, SequenceDiscriminator, JointGenerator
from ultils import nt_xent_loss, kl_divergence, feature_matching_loss, np_rounding
from visualise import visualise_gan, plot_metrics_trend
from metrics import evaluate_all

def compute_gp(dis, r, f, dev):
    a = torch.rand(r.size(0), 1, 1, device=dev)
    it = (a * r + (1 - a) * f).requires_grad_(True)
    with torch.backends.cudnn.flags(enabled=False): do, _ = dis(it)
    gr = torch.autograd.grad(do, it, torch.ones_like(do), True, True)[0].reshape(r.size(0), -1)
    return ((gr.norm(2, 1) - 1) ** 2).mean()

def train_m3gan(ds, cfg, mav, miv):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cv, dv = AutoregressiveVAE(cfg['c_dim'], cfg['hidden_dim'], cfg['latent_dim'], cfg['enc_layers'], cfg['dec_layers'], cfg['time_steps']).to(dev), AutoregressiveVAE(cfg['d_dim'], cfg['hidden_dim'], cfg['latent_dim'], cfg['enc_layers'], cfg['dec_layers'], cfg['time_steps']).to(dev)
    jg = JointGenerator(cfg['noise_dim'], cfg['noise_dim'], cfg['hidden_dim'], cfg['latent_dim'], cfg['latent_dim'], cfg['gen_layers']).to(dev)
    cd, dd = SequenceDiscriminator(cfg['c_dim'], cfg['hidden_dim'], cfg['time_steps'], cfg['dis_layers']).to(dev), SequenceDiscriminator(cfg['d_dim'], cfg['hidden_dim'], cfg['time_steps'], cfg['dis_layers']).to(dev)
    oVp, oV, oG, oD = optim.Adam(list(cv.parameters())+list(dv.parameters()), cfg['v_lr_pre']), optim.Adam(list(cv.parameters())+list(dv.parameters()), cfg['v_lr']), optim.Adam(jg.parameters(), cfg['g_lr']), optim.Adam(list(cd.parameters())+list(dd.parameters()), cfg['d_lr'])
    sc = torch.amp.GradScaler('cuda', enabled=cfg['use_amp'])

    if cfg.get('resume_checkpoint'):
        print(f"Loading checkpoint: {cfg['resume_checkpoint']}")
        ckpt = torch.load(cfg['resume_checkpoint'], map_location=dev)
        if 'c_vae' in ckpt:cv.load_state_dict(ckpt['c_vae'])
        if 'd_vae' in ckpt: dv.load_state_dict(ckpt['d_vae'])
        if 'c_gen' in ckpt:
            print("loaded pretrained gan")
            jg.c_gen.load_state_dict(ckpt['c_gen'])
        if 'd_gen' in ckpt: jg.d_gen.load_state_dict(ckpt['d_gen'])
        if 'c_dis' in ckpt: cd.load_state_dict(ckpt['c_dis'])
        if 'd_dis' in ckpt: dd.load_state_dict(ckpt['d_dis'])

    if not cfg['skip_pretrain']:
        dl = DataLoader(ds, cfg['pre_batch_size'], True, drop_last=True)
        for e in range(cfg['num_pre_epochs']):
            pb = tqdm(dl, desc=f"VAE Pre [{e+1}]")
            for cx, dx in pb:
                cx, dx = cx.to(dev), dx.to(dev)
                oVp.zero_grad()
                with torch.amp.autocast('cuda', enabled=cfg['use_amp']):
                    (cr, cl, cm, cv_, cz), (dr, dl_, dm, dv_, dz) = cv(cx), dv(dx)
                    ls = cfg['alpha_re']*(F.mse_loss(cr,cx)+F.binary_cross_entropy_with_logits(dl_,dx)) + cfg['alpha_kl']*(kl_divergence(cm,cv_)+kl_divergence(dm,dv_)) + cfg['alpha_ct']*nt_xent_loss(cz.view(cz.size(0),-1), dz.view(dz.size(0),-1)) + cfg['alpha_mt']*F.mse_loss(cz,dz)
                sc.scale(ls).backward(); sc.step(oVp); sc.update()

    dl, hi = DataLoader(ds, cfg['batch_size'], True, drop_last=True), {'epochs':[], 'mmd':[], 'rmse':[], 'corr_c':[]}
    for e in range(cfg['num_epochs']):
        pb = tqdm(dl, desc=f"GAN Joint [{e+1}]")
        for cx, dx in pb:
            cx, dx, bs = cx.to(dev), dx.to(dev), cx.size(0)
            for _ in range(cfg['d_rounds']):
                oD.zero_grad(); nc, nd = torch.randn(bs, cfg['time_steps'], cfg['noise_dim'], device=dev), torch.randn(bs, cfg['time_steps'], cfg['noise_dim'], device=dev)
                with torch.no_grad(): fzc, fzd = jg(nc, nd); fcs, _ = cv.reconstruct_decoder(fzc); fds, _ = dv.reconstruct_decoder(fzd)
                with torch.amp.autocast('cuda', enabled=cfg['use_amp']):
                    rlc, rld = cd(cx)[0], dd(dx)[0]; flc, fld = cd(fcs.detach())[0], dd(fds.detach())[0]
                    dl_ = flc.mean()-rlc.mean() + fld.mean()-rld.mean() + 10*(compute_gp(cd,cx,fcs.detach(),dev)+compute_gp(dd,dx,fds.detach(),dev))
                sc.scale(dl_).backward(); sc.step(oD); sc.update()
            oG.zero_grad()
            with torch.amp.autocast('cuda', enabled=cfg['use_amp']):
                fzc, fzd = jg(torch.randn(bs, cfg['time_steps'], cfg['noise_dim'], device=dev), torch.randn(bs, cfg['time_steps'], cfg['noise_dim'], device=dev))
                fcs, _ = cv.reconstruct_decoder(fzc); fds, _ = dv.reconstruct_decoder(fzd); flc, ffc = cd(fcs); fld, ffd = dd(fds)
                with torch.no_grad(): _, rfc = cd(cx); _, rfd = dd(dx)
                gl = -flc.mean()-fld.mean() + cfg['c_beta_fm']*feature_matching_loss(ffc,rfc) + cfg['d_beta_fm']*feature_matching_loss(ffd,rfd)
            sc.scale(gl).backward(); sc.step(oG); sc.update()
            oV.zero_grad()
            with torch.amp.autocast('cuda', enabled=cfg['use_amp']):
                    (cr, cl, cm, cv_, cz), (dr, dl_, dm, dv_, dz) = cv(cx), dv(dx)
                    vl = cfg['alpha_re']*(F.mse_loss(cr,cx)+F.binary_cross_entropy_with_logits(dl_,dx)) + cfg['alpha_kl']*(kl_divergence(cm,cv_)+kl_divergence(dm,dv_)) + cfg['alpha_ct']*nt_xent_loss(cz.view(cz.size(0),-1), dz.view(dz.size(0),-1)) + cfg['alpha_mt']*F.mse_loss(cz,dz)
            sc.scale(vl).backward(); sc.step(oV); sc.update()
        if (e+1) % 5 == 0: print(f"✅ Epoch {e+1} completed.")
        if (e+1) % 50 == 0:
            with torch.no_grad():
                fzc, fzd = jg(torch.randn(1000,cfg['time_steps'],cfg['noise_dim'],device=dev), torch.randn(1000,cfg['time_steps'],cfg['noise_dim'],device=dev))
                fcs, _ = cv.reconstruct_decoder(fzc); fds, _ = dv.reconstruct_decoder(fzd)
                s = evaluate_all(cx.cpu().numpy(), fcs.cpu().numpy(), dx.cpu().numpy(), np_rounding(fds.cpu().numpy()))
            for k in hi: (hi[k].append(s[k]) if k in s else hi['epochs'].append(e+1))
            plot_metrics_trend(hi); visualise_gan(cx.cpu().numpy(), fcs.cpu().numpy(), dx.cpu().numpy(), fds.cpu().numpy(), e+1, mav, miv)
            torch.save({'c_gen':jg.c_gen.state_dict(),'d_gen':jg.d_gen.state_dict(),'c_vae':cv.state_dict(),'d_vae':dv.state_dict()}, f"Output/checkpoint/m3gan_{e+1}.pth")

Overwriting trainer.py


In [7]:
%%writefile main.py
import torch, os, numpy as np, pickle, argparse
from torch.utils.data import TensorDataset
from trainer import train_m3gan
def main(args):
    dp = os.path.join('Data/', args.dataset)
    with open(os.path.join(dp, 'vital_sign_24hrs.pkl'), 'rb') as f: cx = pickle.load(f)
    with open(os.path.join(dp, 'med_interv_24hrs.pkl'), 'rb') as f: dx = pickle.load(f)
    dx, cx = np.nan_to_num(np.clip(dx, 0, 1), 0), np.nan_to_num(cx, 0)
    mi, ma = np.min(cx, axis=(0, 1)), np.max(cx, axis=(0, 1))
    rg = ma - mi; rg[rg==0] = 1e-6; cx = (cx - mi) / rg
    cfg = { 'batch_size':args.batch_size, 'pre_batch_size':args.pre_batch_size, 'time_steps':cx.shape[1], 'c_dim':cx.shape[2], 'd_dim':dx.shape[2], 'latent_dim':25, 'hidden_dim':args.gen_num_units, 'noise_dim':min(cx.shape[2]//2, dx.shape[2]//2), 'enc_layers':args.enc_layers, 'dec_layers':args.dec_layers, 'gen_layers':args.gen_num_layers, 'dis_layers':args.dis_num_layers, 'num_pre_epochs':args.num_pre_epochs, 'num_epochs':args.num_epochs, 'epoch_ckpt_freq':args.epoch_ckpt_freq, 'v_lr_pre':args.v_lr_pre, 'v_lr':args.v_lr, 'g_lr':args.g_lr, 'd_lr':args.d_lr, 'd_rounds':args.d_rounds, 'g_rounds':args.g_rounds, 'v_rounds':args.v_rounds, 'alpha_re':args.alpha_re, 'alpha_kl':args.alpha_kl, 'alpha_mt':args.alpha_mt, 'alpha_ct':args.alpha_ct, 'c_beta_adv':args.c_beta_adv, 'c_beta_fm':args.c_beta_fm, 'd_beta_adv':args.d_beta_adv, 'd_beta_fm':args.d_beta_fm, 'skip_pretrain':args.skip_pretrain, 'use_amp':args.use_amp, 'resume_checkpoint':args.resume_checkpoint }
    train_m3gan(TensorDataset(torch.tensor(cx, dtype=torch.float32), torch.tensor(dx, dtype=torch.float32)), cfg, rg, mi)
if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--dataset', type=str, default="mimic")
    p.add_argument('--batch_size', type=int, default=256)
    p.add_argument('--pre_batch_size', type=int, default=1024)
    p.add_argument('--num_pre_epochs', type=int, default=500)
    p.add_argument('--num_epochs', type=int, default=800)
    p.add_argument('--epoch_ckpt_freq', type=int, default=50)
    p.add_argument('--v_lr_pre', type=float, default=0.0005)
    p.add_argument('--v_lr', type=float, default=0.0001)
    p.add_argument('--g_lr', type=float, default=0.0001)
    p.add_argument('--d_lr', type=float, default=0.0001)
    p.add_argument('--d_rounds', type=int, default=2)
    p.add_argument('--g_rounds', type=int, default=1)
    p.add_argument('--v_rounds', type=int, default=1)
    p.add_argument('--alpha_re', type=float, default=1)
    p.add_argument('--alpha_kl', type=float, default=0.5)
    p.add_argument('--alpha_mt', type=float, default=0.1)
    p.add_argument('--alpha_ct', type=float, default=0.1)
    p.add_argument('--c_beta_adv', type=float, default=1)
    p.add_argument('--c_beta_fm', type=float, default=10.0)
    p.add_argument('--d_beta_adv', type=float, default=1.0)
    p.add_argument('--d_beta_fm', type=float, default=10.0)
    p.add_argument('--enc_layers', type=int, default=3)
    p.add_argument('--dec_layers', type=int, default=3)
    p.add_argument('--gen_num_units', type=int, default=512)
    p.add_argument('--gen_num_layers', type=int, default=3)
    p.add_argument('--dis_num_layers', type=int, default=3)
    p.add_argument('--skip_pretrain', action='store_true', default=True)
    p.add_argument('--use_amp', action='store_true', default=True)
    p.add_argument('--resume_checkpoint', type=str, default=None)
    main(p.parse_args())

Writing main.py


In [11]:
# Start Training!
# Update --resume_checkpoint with your actual path if needed
!python main.py --dataset mimic --skip_pretrain --use_amp --batch_size 128 --epoch_ckpt_freq 1 --resume_checkpoint /kaggle/input/models/lmnhthng/neo-mgan3-epoch100/pytorch/default/1/neo_m3gan_100.pth --num_epochs 150

Loading checkpoint: /kaggle/input/models/lmnhthng/neo-mgan3-epoch100/pytorch/default/1/neo_m3gan_100.pth
loaded pretrained gan
GAN Joint [1]:   0%|                                    | 0/221 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
GAN Joint [1]:   1%|▎                           | 2/221 [00:04<08:22,  2.29s/it]
Traceback (most recent call last):
  File "/kaggle/working/main.py", line 44, in <module>
    main(p.parse_args())
  File "/kaggle/working/main.py", line 12, in main
    train_m3gan(TensorDataset(torch.tensor(cx, dtype=torch.float32), torch.tensor(dx, dtype=torch.float32)), cfg, rg, mi)
  File "/kaggle/working/trainer.py", line 59, in tra